In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.chains.question_answering import load_qa_chain

In [47]:
import os

In [48]:
from getpass import getpass

api_key = getpass("Cole sua OPENAI_API_KEY e pressione Enter: ").strip()
if not api_key:
    raise ValueError("Adicione sua OPENAI_API_KEY para continuar.")

os.environ["OPENAI_API_KEY"] = api_key

ValueError: Adicione sua OPENAI_API_KEY para continuar.

In [ ]:
#load dos modelos
# a API aceita no maximo 300 mil tokens por chamada; 100 trechos ficam bem abaixo disso
embeddings_model = OpenAIEmbeddings(chunk_size=100)
llm = ChatOpenAI(
    model="gpt-6-luna",
    max_completion_tokens=2000,
    reasoning_effort="low",
)

In [ ]:
# carregar o pdf

Pdf_link = "os-sertoes.pdf"
loader = PyPDFLoader(Pdf_link, extract_images=False)

papers = loader.load_and_split()

In [ ]:
# Separar em chunks (pedacos de documentos)
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size=4000, 
    chunk_overlap=20,
    length_function=len,
    add_start_index=True,
)

chunks = text_spliter.split_documents(papers)

In [ ]:
# Salvar no vector DB chorma
vector_store = Chroma.from_documents(
    chunks,
    embedding=embeddings_model,
    persist_directory="db_index"
)

vector_store.persist()

AuthenticationError: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}

In [ ]:
# Carregar o vector DB
vector_store = Chroma(
    embedding_function=embeddings_model,
    persist_directory="db_index"
)

# Load retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# construcao da cadeia de prompts
chain = load_qa_chain(llm=llm, chain_type="stuff")

In [ ]:
def ask(question):
    context = retriever.get_relevant_documents(question)
    answer = (chain({"input_documents": context, "question": question}, return_only_outputs=True))["output_text"]
    return answer

In [ ]:
user_question = input("Digite sua pergunta: ")
answer = ask(user_question)
print("Resposta: ", answer)

Resposta:  Com base no trecho, os principais pontos de atenção são:

- **Transição e previsibilidade:** planejar como a mudança será implementada, com diálogo e tempo para adaptação.
- **Impacto nas empresas:** avaliar se o setor produtivo conseguirá reorganizar jornadas e operações sem prejudicar serviços ou atividades.
- **Contrapartidas do governo:** a senadora citada defendeu apoio aos empregadores que terão custos ou dificuldades para se adaptar.
- **Despesas municipais:** a mudança pode aumentar os gastos das prefeituras, segundo a preocupação mencionada no debate.
- **Jornadas já inferiores a 40 horas:** uma emenda busca esclarecer que a PEC não reduziria essas jornadas ainda mais; se necessário, a carga seria redistribuída.

O trecho não informa os detalhes completos da PEC nem o resultado final da emenda. Portanto, convém confirmar o texto aprovado e as regras de transição antes de tirar conclusões sobre efeitos concretos.
